# Satellite close-approach hackathon — student notebook

Two satellites have a **close approach** when they pass near each other in space.
Your job is to find those events in a catalog of objects, then check that your
answers are accurate.

**Run the cells from top to bottom.**

### What you will do

1. Load the satellite catalog
2. Build objects you can move forward in time ("propagate")
3. Understand the **naive baseline**: check every pair on a coarse time grid
4. Implement **your own** finder in `student_solution.py`
5. Run the same **verifier** on the baseline and on your code

## 0. Setup

**Google Colab:** run the next two cells (install packages, then fetch the project).

**Local:** use the project virtual environment and open this notebook from the repo root.

In [ ]:
import sys
import subprocess

pkgs = [
    "skyfield>=1.48",
    "sgp4>=2.23",
    "numpy>=1.26",
    "scipy>=1.11",
    "plotly>=5.18",
    "pandas>=2.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencies installed.")

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/abensour/collision_detection_hackathon.git"
REPO_BRANCH = "solve"
CLONE_DIR = Path("/content/collision_detection_hackathon")
BUNDLE_NAME = "colab_conjunction_bundle.zip"


def find_project_root():
    candidates = [
        Path.cwd(),
        CLONE_DIR,
        Path.cwd() / "collision_detection_hackathon",
        Path.cwd() / "conjections_hackaton",
        Path.cwd() / "colab_bundle",
        Path("/content/colab_bundle"),
    ]
    for path in candidates:
        if (path / "conjunction_toolkit").exists() and (path / "spacetrack_data.json").exists():
            return path.resolve()
    return None


def looks_like_colab() -> bool:
    return "google.colab" in sys.modules or Path("/content").exists()


ROOT = find_project_root()

if ROOT is None and looks_like_colab():
    try:
        print(f"Cloning {REPO_URL} ({REPO_BRANCH}) …")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)]
        )
        ROOT = find_project_root()
    except Exception as exc:
        print("Git clone failed:", exc)

if ROOT is None and looks_like_colab():
    from google.colab import files

    print(f"Upload {BUNDLE_NAME} (toolkit + catalog).")
    uploaded = files.upload()
    zip_path = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    if zip_path is None:
        raise FileNotFoundError(f"Please upload {BUNDLE_NAME}")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall("/content")
    ROOT = find_project_root()

if ROOT is None:
    raise FileNotFoundError(
        "Could not find conjunction_toolkit/ and spacetrack_data.json. "
        "Open this notebook from the project root, or upload the Colab bundle."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Catalog present:", (ROOT / "spacetrack_data.json").exists())

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys
import time

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from conjunction_toolkit import (
    ConjunctionClaim,
    catalog_to_satellites,
    load_default_catalog,
    plot_pair_with_distance,
    plot_trajectories,
    refine_closest_approach,
    save_html,
    screen_pairs,
    time_grid,
    verify_claim,
)
from evaluate_solution import evaluate_finder

OUTPUT_DIR = ROOT / "examples" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready.")


## 1. Plain-language vocabulary

| Idea | Meaning |
|------|---------|
| **Catalog** | List of satellites / debris with orbital data |
| **Object id (NORAD id)** | Unique number for each object |
| **Propagate** | Compute where an object is at a chosen time |
| **Time window** | How far into the future we look (for example 6 hours) |
| **Time step** | How far we jump between checks (for example 30 minutes) |
| **Close approach** | Two objects come nearer than a distance threshold |
| **Time of closest approach** | The moment they were nearest |
| **Miss distance** | How close they got (kilometers) |
| **Claim** | Your report: two ids + closest time + miss distance |

Positions in this toolkit are in kilometers in a shared Earth-centered frame.

## 2. Load and inspect the catalog

In [ ]:
catalog = load_default_catalog()

print(f"Source file : {catalog.source_path}")
print(f"Objects     : {len(catalog)}")

epochs = [obj.epoch_utc for obj in catalog]
print(f"Data epochs : {min(epochs).isoformat()} → {max(epochs).isoformat()}")

print("\nA few objects:")
for object_id in catalog.ids()[:5]:
    obj = catalog[object_id]
    print(
        f"  id={obj.norad_cat_id:>6}  name={obj.object_name:<24}  "
        f"inclination={obj.inclination_deg:6.2f} deg  "
        f"revs_per_day={obj.mean_motion_rev_per_day:.4f}"
    )

## 3. Build satellites and look at orbits

We turn catalog rows into objects that can be propagated, then draw a short path.

In [ ]:
demo_ids: list[int] = []
for name in ("ISS (ZARYA)", "HST", "CSS (TIANHE)"):
    matches = list(catalog.filter_by_name(name))
    if matches:
        matches.sort(key=lambda obj: obj.epoch_utc, reverse=True)
        demo_ids.append(matches[0].norad_cat_id)

if len(demo_ids) < 2:
    recent_leo = sorted(
        (obj for obj in catalog if obj.mean_motion_rev_per_day > 14),
        key=lambda obj: obj.epoch_utc,
        reverse=True,
    )
    demo_ids = [obj.norad_cat_id for obj in recent_leo[:3]]

demo_ids = demo_ids[:3]
demo_satellites = catalog_to_satellites(catalog, norad_ids=demo_ids)

orbit_start = max(catalog[object_id].epoch_utc for object_id in demo_ids)
orbit_hours = 6
orbit_end = orbit_start + timedelta(hours=orbit_hours)
orbit_step_seconds = 60  # plot with 1-minute samples

orbit_times = time_grid(orbit_start, orbit_end, step_seconds=orbit_step_seconds)
print("Demo objects:", [(object_id, demo_satellites[object_id].name) for object_id in demo_ids])
print(f"Plot window : {orbit_hours} hours")
print(f"From / to   : {orbit_start.isoformat()} → {orbit_end.isoformat()}")
print(f"Plot step   : {orbit_step_seconds} seconds ({len(orbit_times)} samples)")

In [ ]:
fig = plot_trajectories(demo_satellites, orbit_times, title="Sample orbits")
save_html(fig, OUTPUT_DIR / "notebook_trajectories.html")
fig.show()

## 4. Distance between two objects over time

For one pair we can plot how far apart they are, find the nearest moment on the
grid, then **refine** that moment for a better time and distance.

In [ ]:
from conjunction_toolkit import closest_approach_on_grid, propagate_positions
from conjunction_toolkit.propagate import datetimes_of, pair_distances

id_a, id_b = demo_ids[0], demo_ids[1]
sat_a, sat_b = demo_satellites[id_a], demo_satellites[id_b]

# Positions at every sample on the time grid, then distances between the two objects
positions_a = propagate_positions(sat_a, orbit_times)
positions_b = propagate_positions(sat_b, orbit_times)
sample_times = datetimes_of(orbit_times)
distances_km = pair_distances(positions_a, positions_b)

grid_closest_time, grid_closest_distance_km, _ = closest_approach_on_grid(
    positions_a, positions_b, sample_times
)

print(f"Pair {id_a} – {id_b}")
print(f"  Closest on the plot grid : {grid_closest_distance_km:.3f} km at {grid_closest_time.isoformat()}")

closest_time, closest_distance_km = refine_closest_approach(sat_a, sat_b, grid_closest_time)
print(f"  After refinement         : {closest_distance_km:.3f} km at {closest_time.isoformat()}")


## 5. The naive baseline (check every pair)

### What the baseline does

1. Pick a **prediction window** — how many hours into the future we look.
2. Pick a **time step** — how many minutes we jump between position checks.
3. For **every unique pair** of objects:
   - place both objects at each time on the grid
   - measure their distance
   - keep the pair if the smallest distance ≤ threshold

### Numbers used in this example

| Setting | Value | Meaning |
|---------|-------|---------|
| Prediction window | **6 hours** | Look from `window_start` to 6 hours later |
| Time step | **30 minutes** | Jump 30 minutes between checks (`30 * 60` seconds) |
| Distance threshold | **100 km** | Report pairs that come within 100 km on the grid |
| Number of objects | **40** Starlink objects | Keep the demo fast; $N$ objects ⇒ $N(N-1)/2$ pairs |

With a 6-hour window and a 30-minute step there are about
$6 \times 60 / 30 + 1 = 13$ times on the grid per object.

**Trade-off:** a 30-minute jump is easy to understand and fairly fast, but a
real close approach that dips and rises between two samples can be missed or
timed poorly. That is why we refine before verification.

In [ ]:
# --- Baseline experiment settings (edit these to explore) ---
PREDICTION_HOURS = 6
TIME_STEP_MINUTES = 30
TIME_STEP_SECONDS = TIME_STEP_MINUTES * 60
CLOSE_APPROACH_THRESHOLD_KM = 100.0
N_OBJECTS = 40

min_epoch = datetime(2026, 1, 1, tzinfo=timezone.utc)
starlink_pool = [
    obj
    for obj in catalog
    if obj.epoch_utc >= min_epoch and "STARLINK" in obj.object_name.upper()
]
starlink_pool.sort(key=lambda obj: obj.norad_cat_id)
subset = starlink_pool[:N_OBJECTS]
subset_ids = [obj.norad_cat_id for obj in subset]
baseline_satellites = catalog_to_satellites(catalog, norad_ids=subset_ids)

epochs = sorted(obj.epoch_utc for obj in subset)
window_start_utc = epochs[len(epochs) // 2]
window_end_utc = window_start_utc + timedelta(hours=PREDICTION_HOURS)

n_pairs = len(subset_ids) * (len(subset_ids) - 1) // 2
n_time_samples = int(PREDICTION_HOURS * 3600 / TIME_STEP_SECONDS) + 1

print("Baseline setup")
print(f"  Objects                 : {len(subset_ids)}")
print(f"  Unique pairs to check   : {n_pairs}")
print(f"  Prediction window       : {PREDICTION_HOURS} hours")
print(f"  Window start (UTC)      : {window_start_utc.isoformat()}")
print(f"  Window end   (UTC)      : {window_end_utc.isoformat()}")
print(f"  Time step               : {TIME_STEP_MINUTES} minutes  (= {TIME_STEP_SECONDS} seconds)")
print(f"  Samples along the window: ~{n_time_samples}")
print(f"  Distance threshold      : {CLOSE_APPROACH_THRESHOLD_KM} km")

In [ ]:
def naive_baseline_find_close_approaches(
    satellites,
    window_start_utc,
    window_end_utc,
    *,
    time_step_seconds: float,
    close_approach_threshold_km: float,
):
    """Same interface as student_solution.find_close_approaches.

    Uses the toolkit brute-force screener: every pair × every time sample.
    """
    return screen_pairs(
        satellites,
        window_start_utc,
        window_end_utc,
        step_seconds=time_step_seconds,
        threshold_km=close_approach_threshold_km,
        algorithm_id="naive_baseline",
    )


print("Running naive baseline…")
t_run = time.perf_counter()
baseline_raw_claims = naive_baseline_find_close_approaches(
    baseline_satellites,
    window_start_utc,
    window_end_utc,
    time_step_seconds=TIME_STEP_SECONDS,
    close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
)
baseline_seconds = time.perf_counter() - t_run

print(f"Done in {baseline_seconds:.2f} s")
print(f"Close approaches on the {TIME_STEP_MINUTES}-minute grid: {len(baseline_raw_claims)}")
print("\nClosest few (before refinement):")
for claim in baseline_raw_claims[:5]:
    print(
        f"  {claim.norad_a}–{claim.norad_b}: "
        f"{claim.min_distance_km:.3f} km at {claim.tca_utc.isoformat()}"
    )

## 6. Your solution goes in a file

Open **`student_solution.py`** in the project root.

Implement this function (name and arguments must stay the same):

```python
def find_close_approaches(
    satellites,
    window_start_utc,
    window_end_utc,
    *,
    time_step_seconds: float,
    close_approach_threshold_km: float,
):
    ...
```

It must return a list of `ConjunctionClaim` objects. Right now it raises
`NotImplementedError` on purpose.

The next cell only **imports** your file so you can see the stub.

In [ ]:
import importlib
import student_solution

importlib.reload(student_solution)

print("Student entry point:", student_solution.find_close_approaches)
print("File:", Path(student_solution.__file__).resolve())
print()
print("Trying the stub (expected: NotImplementedError)…")
try:
    student_solution.find_close_approaches(
        baseline_satellites,
        window_start_utc,
        window_end_utc,
        time_step_seconds=TIME_STEP_SECONDS,
        close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
    )
except NotImplementedError as exc:
    print("OK — stub not implemented yet:")
    print(" ", exc)

### Workspace for your ideas (optional)

Use the cell below to sketch helpers. When you are ready, copy the real
algorithm into `student_solution.py` so the verifier can import it.

In [ ]:
# --- Optional scratch pad (does not replace student_solution.py) ---
# Example ideas: filter by altitude band, spatial neighbor search, coarser then finer grids, ...

def my_helper_example(satellites):
    """Delete or replace with your own helpers."""
    return list(satellites.keys())


print("Object ids in this demo subset:", len(my_helper_example(baseline_satellites)))

## 7. Verifier block (swap baseline ↔ your code)

`evaluate_finder(...)` always prints the same summary:

- **Close approaches detected** — how many claims the algorithm returned
- **Verified OK** — how many passed independent re-checking
- **Failed verification** — how many did not pass
- **Runtime** — seconds spent inside the finder

Change only the first argument to switch algorithms:

| Call | What runs |
|------|-----------|
| `naive_baseline_find_close_approaches` | Slow check-every-pair baseline |
| `student_solution.find_close_approaches` | Your file |

Verification re-propagates both objects and accepts a claim only if the
miss distance and closest time match within tight tolerances
(default: about 100 meters and 5 seconds).

In [ ]:
# --- Evaluate the NAIVE BASELINE ---
baseline_report = evaluate_finder(
    naive_baseline_find_close_approaches,
    baseline_satellites,
    window_start_utc,
    window_end_utc,
    time_step_seconds=TIME_STEP_SECONDS,
    close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
    algorithm_name="naive_baseline",
    refine_before_verify=True,
)
baseline_report.print_summary()

In [ ]:
# --- Evaluate YOUR SOLUTION (edit student_solution.py first) ---
importlib.reload(student_solution)

try:
    student_report = evaluate_finder(
        student_solution.find_close_approaches,
        baseline_satellites,
        window_start_utc,
        window_end_utc,
        time_step_seconds=TIME_STEP_SECONDS,
        close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
        algorithm_name="student_solution",
        refine_before_verify=True,
    )
    student_report.print_summary()
except NotImplementedError as exc:
    print("Student solution not implemented yet.")
    print(exc)
    student_report = None

In [ ]:
# Side-by-side comparison (runs after both reports exist)
if student_report is not None:
    print("Comparison")
    print(
        f"  {'algorithm':<20} {'detected':>10} {'verified_ok':>12} {'failed':>8} {'seconds':>10}"
    )
    for report in (baseline_report, student_report):
        print(
            f"  {report.algorithm_name:<20} "
            f"{report.close_approaches_detected:>10} "
            f"{report.claims_verified_ok:>12} "
            f"{report.claims_failed:>8} "
            f"{report.runtime_seconds:>10.2f}"
        )
else:
    print("Implement student_solution.py, re-run the student verifier cell, then compare.")

### Optional: plot one verified close approach

In [ ]:
report_to_plot = baseline_report  # or student_report after it works

if report_to_plot.claims:
    best = report_to_plot.claims[0]
    zoom_times = time_grid(
        best.tca_utc - timedelta(minutes=45),
        best.tca_utc + timedelta(minutes=45),
        step_seconds=30.0,
    )
    fig_pair = plot_pair_with_distance(
        baseline_satellites[best.norad_a],
        baseline_satellites[best.norad_b],
        zoom_times,
        tca=best.tca_utc,
        norad_a=best.norad_a,
        norad_b=best.norad_b,
        threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
        title=f"Close approach {best.norad_a}–{best.norad_b}",
    )
    save_html(fig_pair, OUTPUT_DIR / "notebook_pair.html")
    fig_pair.show()
else:
    print("No claims to plot. Try a larger threshold or more objects.")

---

### Quick reference

| Piece | Where |
|-------|-------|
| Your algorithm | `student_solution.py` → `find_close_approaches` |
| Shared scoring | `evaluate_solution.py` → `evaluate_finder` |
| Naive baseline | `naive_baseline_find_close_approaches` in this notebook |
| Claim fields | `norad_a`, `norad_b`, `tca_utc` (time of closest approach), `min_distance_km` |

Good luck — optimize for **correct verified claims** and **runtime**, not only raw detections.